In [1]:
import torch

import torch.nn.functional as F
from transformers import pipeline, AutoModel, AutoTokenizer
from tqdm import tqdm

# ⚙️ Set device to CPU (for all parts)
DEVICE = torch.device("cpu")

# 🔄 Load LaBSE for semantic alignment check
labse_model = AutoModel.from_pretrained("sentence-transformers/LaBSE").to(DEVICE)
labse_tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/LaBSE")

# ✅ Use CPU (device=-1)
pipe = pipeline("translation", model="shhossain/opus-mt-en-to-bn", device=-1)


2025-06-19 13:44:57.061394: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750340697.346998      13 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750340697.431851      13 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


config.json:   0%|          | 0.00/804 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.88G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/5.22M [00:00<?, ?B/s]

...

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/969k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.05M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use cpu


In [2]:
import json
import pandas as pd
from tqdm import tqdm

In [3]:
import json
import pandas as pd

with open('/kaggle/input/coco-image-caption/annotations_trainval2014/annotations/captions_train2014.json', 'r') as f:
    coco_data = json.load(f)

annotations = coco_data['annotations']

# Convert to DataFrame
df_mscoco = pd.DataFrame(annotations)[['id', 'image_id', 'caption']]
df_mscoco.rename(columns={'id': 'caption_id', 'caption': 'caption_en'}, inplace=True)

df_mscoco.head()


,caption_id,image_id,caption_en
0,48,318556,A very clean and well decorated empty bathroom
1,67,116100,A panoramic view of a kitchen and all of its a...
2,126,318556,A blue and white bathroom with butterfly theme...
3,148,116100,A panoramic photo of a kitchen and dining room
4,173,379340,A graffiti-ed stop sign across the street from...


In [4]:
print(f"Total captions: {len(df_mscoco)}")
print(f"Unique images: {df_mscoco['image_id'].nunique()}")

Total captions: 414113
Unique images: 82783


In [5]:
# Sample subset of captions
sample_df = df_mscoco.sample(n=10, random_state=42).reset_index(drop=True)
captions_en = sample_df['caption_en'].tolist()

# Translate in batches to avoid token limits
translated_bn = [pipe(text)[0]['translation_text'] for text in captions_en]

# Store the results
sample_df['caption_bn'] = translated_bn
sample_df[['caption_en', 'caption_bn']]

,caption_en,caption_bn
0,A woman laying on a carpeted floor with a laptop.,একজন মহিলা একটা ল্যাপটপ দিয়ে কার্পেটেড মেঝেতে...
1,A very small cute girl in a some pretty clothes.,একটা সুন্দর কাপড়ের মধ্যে একটা ছোট্ট কিউট মেয়ে।
2,A tall brick wall next to a tall white building.,লম্বা সাদা ভবনের পাশে একটি লম্বা ইটের দেয়াল।
3,a plane getting ready to land on the runway,রানওয়েতে ল্যান্ড করার জন্য একটি বিমান প্রস্তুত
4,A large group of giraffes walking in the tall ...,একদল জিরাফ লম্বা ঘাসে হাঁটছে।
5,Several cows graze in a field while one looks ...,"বেশ কয়েকটা গরু একটা মাঠে খেতে থাকে, যখন কেউ ক..."
6,tHERE ARE MANY MIRRORS IN THIS ROOM WITH FLOWE...,পাখির ওপর ঝোপঝাড়ে অনেক লোক আছেন
7,A pizza pocket is sitting on aluminum foil on ...,একটি পিৎজা পকেটে স্টোভের উপর আলুমিনিয়াম ফাঁকি...
8,"A brown, grey, and white cat sitting on a tv","একটি বাদামি, ধূসর এবং সাদা বিড়াল টিভিতে বসে আছে"
9,A girl is dangling a large object into her wid...,একজন মেয়ে দাঁড়িয়ে থাকা অবস্থায় তার খোলা মু...


In [6]:
# 🧠 Split into 10 parts
total = len(df_mscoco)
batch_size = 128
split_points = [total // 10 * i for i in range(1, 11)]  # Divide into 10 parts

# 🔁 Part 1 (0 to 1/10)
#df_part = df_mscoco.iloc[:split_points[0]].reset_index(drop=True)
# 🔁 Part 2 (1/10 to 2/10)
#df_part = df_mscoco.iloc[split_points[0]:split_points[1]].reset_index(drop=True)
# 🔁 Part 3 (2/10 to 3/10)
#df_part = df_mscoco.iloc[split_points[1]:split_points[2]].reset_index(drop=True)
# 🔁 Part 4 (3/10 to 4/10)
#df_part = df_mscoco.iloc[split_points[2]:split_points[3]].reset_index(drop=True)
# 🔁 Part 5 (4/10 to 5/10)
df_part = df_mscoco.iloc[split_points[3]:split_points[4]].reset_index(drop=True)
# 🔁 Part 6 (5/10 to 6/10)
# df_part = df_mscoco.iloc[split_points[4]:split_points[5]].reset_index(drop=True)
# 🔁 Part 7 (6/10 to 7/10)
# df_part = df_mscoco.iloc[split_points[5]:split_points[6]].reset_index(drop=True)
# 🔁 Part 8 (7/10 to 8/10)
#df_part = df_mscoco.iloc[split_points[6]:split_points[7]].reset_index(drop=True)
# 🔁 Part 9 (8/10 to 9/10)
#df_part = df_mscoco.iloc[split_points[7]:split_points[8]].reset_index(drop=True)
# 🔁 Part 10 (9/10 to end)
#df_part = df_mscoco.iloc[split_points[8]:].reset_index(drop=True)

In [7]:
# ✅ Secure translation function with LaBSE verification
def translate_and_verify_batch(batch_en, similarity_threshold=0.75):
    translations = []
    similarities = []
    validity = []

    for cap in batch_en:
        try:
            # Translate
            cap_bn = pipe(cap)[0]['translation_text']

            # LaBSE semantic similarity
            with torch.no_grad():
                tokens = labse_tokenizer([cap, cap_bn], return_tensors="pt", padding=True, truncation=True).to(DEVICE)
                embeddings = labse_model(**tokens).pooler_output
                emb_en, emb_bn = F.normalize(embeddings[0], dim=-1), F.normalize(embeddings[1], dim=-1)
                sim = torch.dot(emb_en, emb_bn).item()

            translations.append(cap_bn)
            similarities.append(sim)
            validity.append(sim >= similarity_threshold)

        except Exception as e:
            print(f"⚠️ Error with caption: {cap} | {e}")
            translations.append("")
            similarities.append(0.0)
            validity.append(False)

    return translations, validity, similarities

In [8]:
# Sample 10 random captions
sample_df = df_mscoco.sample(n=10, random_state=42).reset_index(drop=True)
captions_en = sample_df['caption_en'].tolist()

# Translate and verify
translated_bn, is_valid, sim_scores = translate_and_verify_batch(captions_en)

# Store results back in DataFrame
sample_df['caption_bn'] = translated_bn
sample_df['semantically_valid'] = is_valid
sample_df['labse_similarity'] = sim_scores

# Print the results
for index, row in sample_df.iterrows():
    print(f"EN: {row['caption_en']}")
    print(f"BN: {row['caption_bn']}")
    print(f"Similarity: {row['labse_similarity']:.3f}")
    print(f"Valid: {'Yes' if row['semantically_valid'] else 'No'}")
    print("-" * 50)

sample_df.head(10)

EN: A woman laying on a carpeted floor with a laptop.
BN: একজন মহিলা একটা ল্যাপটপ দিয়ে কার্পেটেড মেঝেতে শুয়ে আছে।
Similarity: 0.897
Valid: Yes
--------------------------------------------------
EN: A very small cute girl in a some pretty clothes.
BN: একটা সুন্দর কাপড়ের মধ্যে একটা ছোট্ট কিউট মেয়ে।
Similarity: 0.883
Valid: Yes
--------------------------------------------------
EN: A tall brick wall next to a tall white building.
BN: লম্বা সাদা ভবনের পাশে একটি লম্বা ইটের দেয়াল।
Similarity: 0.901
Valid: Yes
--------------------------------------------------
EN: a plane getting ready to land on the runway
BN: রানওয়েতে ল্যান্ড করার জন্য একটি বিমান প্রস্তুত
Similarity: 0.916
Valid: Yes
--------------------------------------------------
EN: A large group of giraffes walking in the tall grass. 
BN: একদল জিরাফ লম্বা ঘাসে হাঁটছে।
Similarity: 0.797
Valid: Yes
--------------------------------------------------
EN: Several cows graze in a field while one looks towards the camera.
BN: বেশ কয়েক

,caption_id,image_id,caption_en,caption_bn,semantically_valid,labse_similarity
0,667999,126073,A woman laying on a carpeted floor with a laptop.,একজন মহিলা একটা ল্যাপটপ দিয়ে কার্পেটেড মেঝেতে...,True,0.896893
1,573347,272716,A very small cute girl in a some pretty clothes.,একটা সুন্দর কাপড়ের মধ্যে একটা ছোট্ট কিউট মেয়ে।,True,0.882576
2,478763,241962,A tall brick wall next to a tall white building.,লম্বা সাদা ভবনের পাশে একটি লম্বা ইটের দেয়াল।,True,0.900876
3,490390,248204,a plane getting ready to land on the runway,রানওয়েতে ল্যান্ড করার জন্য একটি বিমান প্রস্তুত,True,0.915969
4,708276,136168,A large group of giraffes walking in the tall ...,একদল জিরাফ লম্বা ঘাসে হাঁটছে।,True,0.796669
5,187309,479328,Several cows graze in a field while one looks ...,"বেশ কয়েকটা গরু একটা মাঠে খেতে থাকে, যখন কেউ ক...",True,0.859967
6,284987,480831,tHERE ARE MANY MIRRORS IN THIS ROOM WITH FLOWE...,পাখির ওপর ঝোপঝাড়ে অনেক লোক আছেন,False,0.494723
7,602660,510706,A pizza pocket is sitting on aluminum foil on ...,একটি পিৎজা পকেটে স্টোভের উপর আলুমিনিয়াম ফাঁকি...,True,0.809137
8,659590,379612,"A brown, grey, and white cat sitting on a tv","একটি বাদামি, ধূসর এবং সাদা বিড়াল টিভিতে বসে আছে",True,0.896823
9,73793,525823,A girl is dangling a large object into her wid...,একজন মেয়ে দাঁড়িয়ে থাকা অবস্থায় তার খোলা মু...,True,0.844836


In [9]:
# 🚀 Translate and verify in batches
all_bn, all_valid, all_sims = [], [], []

for i in tqdm(range(0, len(df_part), batch_size)):
    batch = df_part['caption_en'].iloc[i:i+batch_size].tolist()
    bn, valid, sims = translate_and_verify_batch(batch)
    all_bn.extend(bn)
    all_valid.extend(valid)
    all_sims.extend(sims)

100%|██████████| 324/324 [7:41:19<00:00, 85.43s/it]


In [10]:
# 📄 Save result
df_part['caption_bn'] = all_bn
df_part['semantically_valid'] = all_valid
df_part['labse_similarity'] = all_sims

# ✂️ Save each split separately
# df_part.to_csv(f"/kaggle/working/secure_translated_bn_part1.csv", index=False)
# df_part.to_csv(f"/kaggle/working/secure_translated_bn_part2.csv", index=False)
# df_part.to_csv(f"/kaggle/working/secure_translated_bn_part3.csv", index=False)
# df_part.to_csv(f"/kaggle/working/secure_translated_bn_part4.csv", index=False)
df_part.to_csv(f"/kaggle/working/secure_translated_bn_part5.csv", index=False)
# df_part.to_csv(f"/kaggle/working/secure_translated_bn_part6.csv", index=False)
# df_part.to_csv(f"/kaggle/working/secure_translated_bn_part7.csv", index=False)
# df_part.to_csv(f"/kaggle/working/secure_translated_bn_part8.csv", index=False)
# df_part.to_csv(f"/kaggle/working/secure_translated_bn_part9.csv", index=False)
# df_part.to_csv(f"/kaggle/working/secure_translated_bn_part10.csv", index=False)